In [1]:
import os
import json
import hashlib
import requests
import pandas as pd
import time
from datetime import datetime
from datetime import timedelta
from dotenv import load_dotenv

import sys
sys.path.append('..')
from utils.bucket_utils import get_duck_con

In [2]:
file_path = "../source_list/pathum_loction.json"

if os.path.exists(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        locations = json.load(f)
    print(f"✅ โหลดพิกัดจากไฟล์ {file_path} จำนวน {len(locations)} พื้นที่")
else:
    print(f"❌ ไม่พบไฟล์ {file_path} กรุณาสร้างไฟล์ข้อมูลก่อนรันโปรแกรม")
    locations = [] 

✅ โหลดพิกัดจากไฟล์ ../source_list/pathum_loction.json จำนวน 60 พื้นที่


In [12]:
def generate_log_id(timestamp, tambon):
    """
    สร้าง Primary Key คงที่จากเวลาและสถานที่
    ป้องกันปัญหาข้อมูลเบิ้ล (Duplicates) เวลารัน Pipeline ซ้ำใน Prefect
    """
    unique_string = f"{timestamp}_{tambon}"
    return hashlib.md5(unique_string.encode('utf-8')).hexdigest()

def get_weather_condition(wmo_code):
    """แปลงรหัส WMO Code เป็นข้อความสภาพอากาศ (ครบทุก code ที่ Open-Meteo ใช้)"""
    weather_mapping = {
        # ท้องฟ้าและเมฆ
        0:  "ท้องฟ้าแจ่มใส",
        1:  "แจ่มใสเป็นส่วนใหญ่",
        2:  "มีเมฆบางส่วน",
        3:  "เมฆมาก",
        # หมอก
        45: "หมอก",
        48: "หมอกเกาะน้ำแข็ง",
        # ฝนละออง
        51: "ฝนละออง: เบา",
        53: "ฝนละออง: ปานกลาง",
        55: "ฝนละออง: หนาแน่น",
        # ฝนละอองเยือกแข็ง
        56: "ฝนละอองเยือกแข็ง: เบา",
        57: "ฝนละอองเยือกแข็ง: หนาแน่น",
        # ฝน
        61: "ฝน: เล็กน้อย",
        63: "ฝน: ปานกลาง",
        65: "ฝน: หนัก",
        # ฝนเยือกแข็ง
        66: "ฝนเยือกแข็ง: เบา",
        67: "ฝนเยือกแข็ง: หนัก",
        # หิมะ
        71: "หิมะ: เบา",
        73: "หิมะ: ปานกลาง",
        75: "หิมะ: หนัก",
        77: "เกล็ดหิมะ",
        # ฝนตกเป็นช่วง
        80: "ฝนตกเป็นช่วง: เบา",
        81: "ฝนตกเป็นช่วง: ปานกลาง",
        82: "ฝนตกเป็นช่วง: หนักมาก",
        # หิมะตกเป็นช่วง
        85: "หิมะตกเป็นช่วง: เบา",
        86: "หิมะตกเป็นช่วง: หนัก",
        # พายุฝนฟ้าคะนอง
        95: "พายุฝนฟ้าคะนอง",
        96: "พายุฝนฟ้าคะนองมีลูกเห็บเล็กน้อย",
        99: "พายุฝนฟ้าคะนองมีลูกเห็บหนัก",
    }
    return weather_mapping.get(wmo_code, "ไม่ทราบสภาพอากาศ")

def check_outage_risk(wind_speed, wind_gust, precipitation, wmo_code):
    """
    ประเมินความเสี่ยงไฟดับ (Outage Risk) ปรับปรุงใหม่
    เงื่อนไขที่ทำให้เสี่ยงไฟดับ: 
    - ลมเฉลี่ย > 40 km/h
    - ลมกระโชก (สำคัญมาก!) > 60 km/h (อาจพัดกิ่งไม้โดนสายไฟ)
    - ฝนตกหนักมาก > 15 mm
    - มีพายุฟ้าคะนองรุนแรง (รหัส 95, 96, 99)
    """
    if pd.isna(wind_speed) or pd.isna(precipitation) or pd.isna(wmo_code):
        return False
        
    gust = wind_gust if not pd.isna(wind_gust) else 0.0
        
    if wind_speed >= 40.0 or gust >= 60.0 or precipitation >= 15.0 or wmo_code in [95, 96, 99]:
        return True
    return False

def save_to_gcs_parquet(final_df, path):
    """
    บันทึก DataFrame เป็นไฟล์ .parquet ลง GCS โดยใช้ DuckDB 
    """
    if final_df is not None and not final_df.empty:
        print(f"🗄️ กำลังบันทึกข้อมูล {len(final_df)} แถว ลง {path}...")
        
        con = get_duck_con()
        try:
            # DuckDB สามารถอ่านตัวแปร DataFrame ชื่อ final_df ได้โดยตรง
            con.execute(f"""
                COPY (
                    SELECT *
                    FROM final_df
                )
                TO '{path}' (FORMAT PARQUET)
            """)
            print(f"✅ บันทึกไฟล์ Parquet สำเร็จ!")
        except Exception as e:
            print(f"❌ เกิดข้อผิดพลาดในการสร้างไฟล์ Parquet: {e}")
        finally:
            con.close()
    else:
        print("⚠️ ไม่มีข้อมูลให้บันทึก")


In [13]:
def fetch_historical_schema_data(target_locations, start_date="2023-01-01", end_date="2025-12-31"):
    archive_url = "https://archive-api.open-meteo.com/v1/archive"
    all_schema_data = []
    
    print(f"⏳ กำลังเริ่มดึงข้อมูลย้อนหลังทีละตำบล (ตั้งแต่ {start_date} ถึง {end_date})...")

    for loc in target_locations:
        print(f"กำลังดึงข้อมูล: {loc['tambon']} ...", end=" ")
        
        # เพิ่ม apparent_temperature และ wind_gusts_10m สำหรับวิเคราะห์ Outage
        weather_params = {
            "latitude": loc["lat"],
            "longitude": loc["lon"],
            "start_date": start_date,
            "end_date": end_date,
            "hourly": "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,weather_code,wind_speed_10m,wind_gusts_10m",
            "timezone": "Asia/Bangkok"
        }
        
        try:
            w_res = requests.get(archive_url, params=weather_params)
            w_res.raise_for_status()
            w_data = w_res.json()
            w_hourly = w_data.get("hourly", {})
            
            if "time" in w_hourly:
                times = w_hourly["time"]
                
                for i in range(len(times)):
                    temp = w_hourly.get("temperature_2m", [])[i] if "temperature_2m" in w_hourly else None
                    feels_like = w_hourly.get("apparent_temperature", [])[i] if "apparent_temperature" in w_hourly else None
                    rh = w_hourly.get("relative_humidity_2m", [])[i] if "relative_humidity_2m" in w_hourly else None
                    precip = w_hourly.get("precipitation", [])[i] if "precipitation" in w_hourly else None
                    w_code = w_hourly.get("weather_code", [])[i] if "weather_code" in w_hourly else None
                    wind_speed = w_hourly.get("wind_speed_10m", [])[i] if "wind_speed_10m" in w_hourly else None
                    wind_gust = w_hourly.get("wind_gusts_10m", [])[i] if "wind_gusts_10m" in w_hourly else None
                    
                    record = {
                        "weather_log_id": generate_log_id(times[i], loc["tambon"]),
                        "province": loc["province"],
                        "district": loc["amphoe"],
                        "sub-district": loc["tambon"],
                        "timestamp": times[i],
                        "weather_condition": get_weather_condition(w_code) if w_code is not None else "Unknown",
                        "Temperature": temp,
                        "feels_like_temp": feels_like,  # สะท้อนโหลดของแอร์
                        "Relative Humidity": rh,
                        "wind_speed_kmh": wind_speed,
                        "wind_gusts_kmh": wind_gust,    # ปัจจัยหลักกิ่งไม้หัก
                        "precipitation_mm": precip,
                        "outage_risk_flag": check_outage_risk(wind_speed, wind_gust, precip, w_code)
                    }
                    all_schema_data.append(record)
                    
            print("✅ สำเร็จ")
            time.sleep(1) # ป้องกัน Rate Limit 
            
        except Exception as e:
            print(f"❌ เกิดข้อผิดพลาดกับ {loc['tambon']}: {e}")

    if all_schema_data:
        final_df = pd.DataFrame(all_schema_data)
        path = f's3://pea-oms/landing/weather_data/{start_date}.parquet'
        save_to_gcs_parquet(final_df=final_df, path=path)
    return None

In [15]:
fetch_historical_schema_data(target_locations=locations, start_date='2024-01-01', end_date='2024-12-31')

⏳ กำลังเริ่มดึงข้อมูลย้อนหลังทีละตำบล (ตั้งแต่ 2024-01-01 ถึง 2024-12-31)...
กำลังดึงข้อมูล: บางปรอก ... ✅ สำเร็จ
กำลังดึงข้อมูล: บ้านใหม่ ... ✅ สำเร็จ
กำลังดึงข้อมูล: บ้านกลาง ... ✅ สำเร็จ
กำลังดึงข้อมูล: บ้านฉาง ... ✅ สำเร็จ
กำลังดึงข้อมูล: บ้านกระแชง ... ✅ สำเร็จ
กำลังดึงข้อมูล: บางขะแยง ... ✅ สำเร็จ
กำลังดึงข้อมูล: บางคูวัด ... ✅ สำเร็จ
กำลังดึงข้อมูล: บางหลวง ... ✅ สำเร็จ
กำลังดึงข้อมูล: บางเดื่อ ... ✅ สำเร็จ
กำลังดึงข้อมูล: บางพูด ... ✅ สำเร็จ
กำลังดึงข้อมูล: บางพูน ... ✅ สำเร็จ
กำลังดึงข้อมูล: บางกะดี ... ✅ สำเร็จ
กำลังดึงข้อมูล: สวนพริกไทย ... ✅ สำเร็จ
กำลังดึงข้อมูล: หลักหก ... ✅ สำเร็จ
กำลังดึงข้อมูล: คลองหนึ่ง ... ✅ สำเร็จ
กำลังดึงข้อมูล: คลองสอง ... ✅ สำเร็จ
กำลังดึงข้อมูล: คลองสาม ... ✅ สำเร็จ
กำลังดึงข้อมูล: คลองสี่ ... ✅ สำเร็จ
กำลังดึงข้อมูล: คลองห้า ... ✅ สำเร็จ
กำลังดึงข้อมูล: คลองหก ... ✅ สำเร็จ
กำลังดึงข้อมูล: คลองเจ็ด ... ✅ สำเร็จ
กำลังดึงข้อมูล: ประชาธิปัตย์ ... ✅ สำเร็จ
กำลังดึงข้อมูล: บึงยี่โถ ... ✅ สำเร็จ
กำลังดึงข้อมูล: รังสิต ... ✅ สำเร็จ
กำลังดึงข้อมูล: ลำผั

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ บันทึกไฟล์ Parquet สำเร็จ!


In [20]:
# Sample query ตัวอย่างการดึงข้อมูลโดยเอาเฉพาะค่าที่อัปเดตล่าสุดของแต่ละเวลาและตำบล
con = get_duck_con()
df = con.execute("""
SELECT * EXCLUDE (filename)
    FROM read_parquet('s3://pea-oms/landing/weather_data/**/*.parquet', filename=true)
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY weather_log_id 
        ORDER BY filename DESC
    ) = 1
"""
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [21]:
df

,weather_log_id,province,district,sub-district,timestamp,weather_condition,Temperature,feels_like_temp,Relative Humidity,wind_speed_kmh,wind_gusts_kmh,precipitation_mm,outage_risk_flag
0,0000abb573c08d095f54d29710ee694f,ปทุมธานี,คลองหลวง,คลองหนึ่ง,2023-05-21T07:00,เมฆมาก,29.2,34.1,71,5.8,14.8,0.0,False
1,0000c893905b4093fb024dbb3624955b,ปทุมธานี,ธัญบุรี,ประชาธิปัตย์,2024-03-04T06:00,ท้องฟ้าแจ่มใส,26.4,32.3,93,6.6,14.0,0.0,False
2,00010fd0f7f4fcdf93aaa3a4e3a25a10,ปทุมธานี,ลำลูกกา,บึงคอไห,2024-04-01T01:00,ท้องฟ้าแจ่มใส,28.6,33.8,81,10.5,19.1,0.0,False
3,0001624e3b00715b97dbef4542780bf5,ปทุมธานี,สามโคก,กระแชง,2025-06-05T02:00,เมฆมาก,26.5,32.1,90,6.0,13.0,0.0,False
4,00024b303933eb44ed8401f0ee2ac740,ปทุมธานี,เมืองปทุมธานี,บ้านกลาง,2025-12-26T04:00,เมฆมาก,23.7,24.9,68,10.3,18.7,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1349851,e092d1a3617e8f8ee894e948c7b2e765,ปทุมธานี,ธัญบุรี,รังสิต,2023-10-09T04:00,เมฆมาก,24.9,30.5,96,4.2,6.5,0.0,False
1349852,e093a2a573bdc8d7c47d93614f9fa6a4,ปทุมธานี,ธัญบุรี,บึงยี่โถ,2024-01-07T17:00,ท้องฟ้าแจ่มใส,31.0,34.5,53,3.1,14.4,0.0,False
1349853,e0943a0357e8d9752674c6ae18a87208,ปทุมธานี,สามโคก,ท้ายเกาะ,2025-03-15T14:00,แจ่มใสเป็นส่วนใหญ่,35.8,42.1,46,2.0,15.8,0.0,False
1349854,e094772475b70273705d24cc38bb55d0,ปทุมธานี,คลองหลวง,คลองสาม,2025-12-19T14:00,ท้องฟ้าแจ่มใส,31.5,34.0,45,10.2,25.6,0.0,False


In [ ]:
# Check data each year
con.execute("""
    WITH deduped AS (
        SELECT YEAR(timestamp::TIMESTAMP) AS year
        FROM read_parquet('s3://pea-oms/landing/weather_data/**/*.parquet', filename=true)
        QUALIFY ROW_NUMBER() OVER (PARTITION BY weather_log_id ORDER BY filename DESC) = 1
    )
    SELECT year, COUNT(*) AS total
    FROM deduped
    GROUP BY year
    ORDER BY year
""").df()

,year,total
0,2023,525600
1,2024,298656
2,2025,525600
